# fase_4 - script_hanif Migration

This notebook handles migration of database from old DB to new DB for fase 4.

**Purpose**: Migrasi data Siswa dan Mitra dengan Mapping Kolom Spesifik

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
from datetime import datetime
import pickle
import json
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
config = get_db_config()
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f"Connected to {config['db_old']['database']} and {config['db_new']['database']}")

Connected to dataleap_v5_example and dataleap_v5_migration


## 2. Ambil Data dari DB Lama

In [3]:
hanif_tables_map = [
    ('siswa', 'siswa'),
    ('siswa_keluar', 'siswa_keluar'),
    ('mitra', 'mitra'),
    ('mitra_note', 'mitra_progres'),
    ('mitra_users', 'kemitraan_verifikator'),
    ('siswamitra', 'siswa_mitra'),
    ('siswa_keluar_mitra', 'siswa_mitra_keluar')
]

raw_data = {}
for old_t, new_t in hanif_tables_map:
    try:
        cursor_old.execute(f"SELECT * FROM `{old_t}`")
        raw_data[old_t] = cursor_old.fetchall()
        print(f"✅ {old_t} loaded: {len(raw_data[old_t])} records")
    except Exception as e:
        print(f"❌ ERROR loading {old_t}: {e}")

✅ siswa loaded: 1469 records
✅ siswa_keluar loaded: 556 records
✅ mitra loaded: 22 records
✅ mitra_note loaded: 296 records
✅ mitra_users loaded: 228 records
✅ siswamitra loaded: 0 records
✅ siswa_keluar_mitra loaded: 0 records


## 3. Transform Data (Mapping Berdasarkan hanif_mapping.md)

In [4]:
transformed_dfs = {}

# 1. siswa -> siswa
if 'siswa' in raw_data:
    df = pd.DataFrame(raw_data['siswa'])
    mapping = {
        'idsiswa': 'id_siswa', 'tgl_daftar': 'tanggal_registrasi', 'domisili': 'domisili',
        'nama_lengkap': 'nama_lengkap', 'panggilan': 'nama_panggilan', 'jkel': 'jenis_kelamin',
        'nama_sekolah': 'asal_sekolah', 'level_sekolah': 'tingkat_sekolah', 'nama_ortu': 'nama_orang_tua',
        'pekerjaan_ortu': 'pekerjaan_orang_tua', 'tmp_lahir': 'tempat_lahir', 'tgl_lahir': 'tanggal_lahir',
        'no_induk': 'nomor_induk', 'email': 'email', 'idcalon': 'id_calon',
        'provinsi': 'id_provinsi', 'kabupaten': 'id_kabupaten', 'kecamatan': 'id_kecamatan',
        'kelurahan': 'id_kelurahan', 'idmitra': 'id_mitra', 'nisn': 'nisn', 'nik': 'nik',
        'kewarganegaraan': 'kewarganegaraan', 'agama': 'agama', 'rt': 'rt', 'rw': 'rw',
        'kodepos': 'kode_pos', 'statussiswa': 'status_aktif', 'rekomen': 'rekomendasi',
        'info': 'sumber_info', 'pembayaran': 'metode_pembayaran', 'nama_ayah': 'nama_ayah',
        'pekerjaan_ayah': 'pekerjaan_ayah', 'jenjang_ayah': 'pendidikan_ayah', 
        'penghasilan_ayah': 'penghasilan_ayah', 'nama_ibu': 'nama_ibu', 'penghasilan_ibu': 'penghasilan_ibu',
        'jenjang_ibu': 'pendidikan_ibu', 'nama_wali': 'nama_wali', 'pekerjaan_wali': 'pekerjaan_wali',
        'jenjang_wali': 'pendidikan_wali', 'penghasilan_wali': 'penghasilan_wali',
        'wapeserta': 'wa_siswa', 'wawalmur': 'wa_ortu', 'waadmin': 'wa_administrasi',
        'sts_pengisian': 'status_pengisian', 'bukti': 'path_bukti_bayar', 'lulus': 'status_lulus_siswa',
        'created_bukti': 'tanggal_upload_bukti'
    }
    df = df.rename(columns=mapping)
    df['pekerjaan_ibu'] = None
    df['deleted_at'] = None
    target_cols = [c for c in list(mapping.values()) if c in df.columns] + ['pekerjaan_ibu', 'deleted_at']
    transformed_dfs['siswa'] = df[target_cols]

# 2. kursus_siswa (Source: -)
transformed_dfs['kursus_siswa'] = pd.DataFrame()

# 3. siswa_keluar -> siswa_keluar
if 'siswa_keluar' in raw_data:
    df = pd.DataFrame(raw_data['siswa_keluar'])
    mapping = {
        'idsiswa_keluar': 'id_keluar', 'idsiswa': 'id_siswa',
        'alasan': 'alasan_keluar', 'tanggal': 'tanggal_keluar'
    }
    df = df.rename(columns=mapping)
    df['id_kursus'] = None
    df['id_tag_keluar'] = None
    transformed_dfs['siswa_keluar'] = df[list(mapping.values()) + ['id_kursus', 'id_tag_keluar']]

# 4. mitra -> mitra
if 'mitra' in raw_data:
    df = pd.DataFrame(raw_data['mitra'])
    mapping = {
        'idmitra': 'id_mitra', 'nama': 'nama_mitra', 'instansi': 'nama_instansi',
        'namasekolah': 'nama_sekolah', 'lokasi': 'alamat_mitra', 'kepsek': 'nama_pimpinan',
        'cp': 'kontak_mitra', 'status': 'status_mitra', 'visimisi': 'visi_misi',
        'program': 'program_mitra', 'sdm': 'info_sdm', 'weakness': 'info_kelemahan',
        'rekomen': 'rekomendasi_program', 'jenis': 'jenis_mitra', 'provinsi': 'provinsi_id',
        'kotkab': 'kabupaten_id', 'jml': 'jumlah_siswa_mitra', 'bidang': 'bidang_usaha',
        'leapverse': 'is_leapverse', 'kemitraan': 'status_kemitraan', 'tahun': 'tahun_bergabung',
        'jeniskemitraan': 'tipe_kerjasama', 'elsa': 'is_elsa', 'classin': 'is_classin',
        'mitraleap': 'is_mitra_leap', 'created_at': 'created_at'
    }
    df = df.rename(columns=mapping)
    df['kode_mitra'] = None
    transformed_dfs['mitra'] = df[list(mapping.values()) + ['kode_mitra']]

# 5. mitra_note -> mitra_progres
if 'mitra_note' in raw_data:
    df = pd.DataFrame(raw_data['mitra_note'])
    mapping = {
        'idmnote': 'id_progres_mitra', 'idmitra': 'id_mitra',
        'note': 'catatan_progres_mitra', 'idusers': 'id_user', 'status': 'status_progres_mitra',
        'startdate': 'kemitraan_mulai', 'enddate': 'kemitraan_berakhir', 'created_at': 'created_at'
    }
    transformed_dfs['mitra_progres'] = df.rename(columns=mapping)[list(mapping.values())]

# 6. mitra_users -> kemitraan_verifikator
if 'mitra_users' in raw_data:
    df = pd.DataFrame(raw_data['mitra_users'])
    mapping = {
        'idmusers': 'id_kemitraan', 'idmnote': 'id_progres_mitra', 'idusers': 'id_user'
    }
    transformed_dfs['kemitraan_verifikator'] = df.rename(columns=mapping)[list(mapping.values())]

# 7. siswamitra -> siswa_mitra
if 'siswamitra' in raw_data:
    df = pd.DataFrame(raw_data['siswamitra'])
    mapping = {
        'idsiswa': 'id_sm', 'tgl_daftar': 'tanggal_daftar', 'domisili': 'alamat_domisili',
        'nama_lengkap': 'nama_lengkap', 'panggilan': 'nama_panggilan', 'jkel': 'jenis_kelamin',
        'nama_instansi': 'nama_instansi', 'level_sekolah': 'tingkat_sekolah',
        'pekerjaan': 'pekerjaan_sm', 'tmp_lahir': 'tempat_lahir', 'tgl_lahir': 'tanggal_lahir',
        'no_induk': 'nomor_induk_sm', 'email': 'email_sm', 'tlp': 'wa_sm',
        'keluar': 'status_keluar_sm', 'idmitra': 'id_mitra'
    }
    df = df.rename(columns=mapping)
    df['sertifikat_sm'] = None
    target_cols = list(mapping.values()) + ['sertifikat_sm']
    transformed_dfs['siswa_mitra'] = df.reindex(columns=target_cols)

# 8. siswa_keluar_mitra -> siswa_mitra_keluar
if 'siswa_keluar_mitra' in raw_data:
    df = pd.DataFrame(raw_data['siswa_keluar_mitra'])
    mapping = {
        'idsiswa_keluar': 'id_sm_keluar', 'idsiswa': 'id_sm',
        'alasan': 'alasan_keluar_sm', 'tanggal': 'tanggal_keluar_sm'
    }
    transformed_dfs['siswa_mitra_keluar'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

print(f"✓ Transformasi {len(transformed_dfs)} tabel Fase 4 selesai.")

✓ Transformasi 8 tabel Fase 4 selesai.


## 3.1 Verifikasi Hasil Transformasi
Bagian ini menampilkan perbandingan jumlah data dan tipe data untuk pengecekan manual.

In [5]:
# 1. Ringkasan Jumlah Baris
print("📊 RINGKASAN MIGRASI (RECORDS COUNT)")
print("="*70)
summary_list = []
for old_t, new_t in hanif_tables_map:
    old_c = len(raw_data.get(old_t, []))
    new_c = len(transformed_dfs.get(new_t, []))
    summary_list.append({
        'Tabel Lama': old_t,
        'Tabel Baru': new_t,
        'Old Recs': old_c,
        'New Recs': new_c,
        'Diff': new_c - old_c,
        'Status': "✅ OK" if old_c == new_c else "⚠️ Cek"
    })
display(pd.DataFrame(summary_list))

total_old_all = sum(len(records) for records in raw_data.values())
total_new_all = sum(len(df) for df in transformed_dfs.values())
print(f"\n📢 TOTAL REKAPITULASI: {total_old_all} (Old) ➔ {total_new_all} (New)")
if total_old_all == total_new_all: print("✅ SEMUA DATA TERANGKUT")
else: print(f"⚠️ ADA SELISIH: {total_new_all - total_old_all} baris")

# 2. Detail Perbandingan Kolom & Tipe Data (Side-by-Side)
print("\n🔍 PERBANDINGAN TIPE DATA SIDE-BY-SIDE")
for old_t, new_t in hanif_tables_map:
    print(f"\n{'='*15} {old_t.upper()} ➔ {new_t.upper()} {'='*15}")
    
    df_old = pd.DataFrame(raw_data.get(old_t, []))
    df_new = transformed_dfs.get(new_t, pd.DataFrame())
    
    if not df_new.empty or not df_old.empty:
        comparison = []
        table_mapping = {}
        if old_t == 'siswa': table_mapping = {'idsiswa': 'id_siswa', 'tgl_daftar': 'tanggal_registrasi', 'domisili': 'domisili', 'nama_lengkap': 'nama_lengkap', 'panggilan': 'nama_panggilan', 'jkel': 'jenis_kelamin', 'nama_sekolah': 'asal_sekolah', 'level_sekolah': 'tingkat_sekolah', 'nama_ortu': 'nama_orang_tua', 'pekerjaan_ortu': 'pekerjaan_orang_tua', 'tmp_lahir': 'tempat_lahir', 'tgl_lahir': 'tanggal_lahir', 'no_induk': 'nomor_induk', 'email': 'email', 'idcalon': 'id_calon', 'provinsi': 'id_provinsi', 'kabupaten': 'id_kabupaten', 'kecamatan': 'id_kecamatan', 'kelurahan': 'id_kelurahan', 'idmitra': 'id_mitra', 'nisn': 'nisn', 'nik': 'nik', 'kewarganegaraan': 'kewarganegaraan', 'agama': 'agama', 'rt': 'rt', 'rw': 'rw', 'kodepos': 'kode_pos', 'statussiswa': 'status_aktif', 'rekomen': 'rekomendasi', 'info': 'sumber_info', 'pembayaran': 'metode_pembayaran', 'nama_ayah': 'nama_ayah', 'pekerjaan_ayah': 'pekerjaan_ayah', 'jenjang_ayah': 'pendidikan_ayah', 'penghasilan_ayah': 'penghasilan_ayah', 'nama_ibu': 'nama_ibu', 'penghasilan_ibu': 'penghasilan_ibu', 'jenjang_ibu': 'pendidikan_ibu', 'nama_wali': 'nama_wali', 'pekerjaan_wali': 'pekerjaan_wali', 'jenjang_wali': 'pendidikan_wali', 'penghasilan_wali': 'penghasilan_wali', 'wapeserta': 'wa_siswa', 'wawalmur': 'wa_ortu', 'waadmin': 'wa_administrasi', 'sts_pengisian': 'status_pengisian', 'bukti': 'path_bukti_bayar', 'lulus': 'status_lulus_siswa', 'created_bukti': 'tanggal_upload_bukti'}
        elif old_t == 'siswa_keluar': table_mapping = {'idsiswa_keluar': 'id_keluar', 'idsiswa': 'id_siswa', 'alasan': 'alasan_keluar', 'tanggal': 'tanggal_keluar'}
        elif old_t == 'mitra': table_mapping = {'idmitra': 'id_mitra', 'nama': 'nama_mitra', 'instansi': 'nama_instansi', 'namasekolah': 'nama_sekolah', 'lokasi': 'alamat_mitra', 'kepsek': 'nama_pimpinan', 'cp': 'kontak_mitra', 'status': 'status_mitra', 'visimisi': 'visi_misi', 'program': 'program_mitra', 'sdm': 'info_sdm', 'weakness': 'info_kelemahan', 'rekomen': 'rekomendasi_program', 'jenis': 'jenis_mitra', 'provinsi': 'provinsi_id', 'kotkab': 'kabupaten_id', 'jml': 'jumlah_siswa_mitra', 'bidang': 'bidang_usaha', 'leapverse': 'is_leapverse', 'kemitraan': 'status_kemitraan', 'tahun': 'tahun_bergabung', 'jeniskemitraan': 'tipe_kerjasama', 'elsa': 'is_elsa', 'classin': 'is_classin', 'mitraleap': 'is_mitra_leap', 'created_at': 'created_at'}
        elif old_t == 'mitra_note': table_mapping = {'idmnote': 'id_progres_mitra', 'idmitra': 'id_mitra', 'note': 'catatan_progres_mitra', 'idusers': 'id_user', 'status': 'status_progres_mitra', 'startdate': 'kemitraan_mulai', 'enddate': 'kemitraan_berakhir', 'created_at': 'created_at'}
        elif old_t == 'mitra_users': table_mapping = {'idmusers': 'id_kemitraan', 'idmnote': 'id_progres_mitra', 'idusers': 'id_user'}
        elif old_t == 'siswamitra': table_mapping = {'idsiswa': 'id_sm', 'tgl_daftar': 'tanggal_daftar', 'domisili': 'alamat_domisili', 'nama_lengkap': 'nama_lengkap', 'panggilan': 'nama_panggilan', 'jkel': 'jenis_kelamin', 'nama_instansi': 'nama_instansi', 'level_sekolah': 'tingkat_sekolah', 'pekerjaan': 'pekerjaan_sm', 'tmp_lahir': 'tempat_lahir', 'tgl_lahir': 'tanggal_lahir', 'no_induk': 'nomor_induk_sm', 'email': 'email_sm', 'tlp': 'wa_sm', 'keluar': 'status_keluar_sm', 'idmitra': 'id_mitra'}
        elif old_t == 'siswa_keluar_mitra': table_mapping = {'idsiswa_keluar': 'id_sm_keluar', 'idsiswa': 'id_sm', 'alasan': 'alasan_keluar_sm', 'tanggal': 'tanggal_keluar_sm'}

        for old_col, new_col in table_mapping.items():
            comparison.append({
                'Old Column': old_col,
                'Old Type': str(df_old[old_col].dtype) if not df_old.empty and old_col in df_old.columns else "N/A",
                '➔': '➔',
                'New Column': new_col,
                'New Type': str(df_new[new_col].dtype) if not df_new.empty and new_col in df_new.columns else "N/A"
            })
        
        # Cek kolom baru
        if not df_new.empty:
            for col in df_new.columns:
                if col not in table_mapping.values():
                    comparison.append({
                        'Old Column': '(KOLOM BARU / CUSTOM)',
                        'Old Type': '-',
                        '➔': '➔',
                        'New Column': col,
                        'New Type': str(df_new[col].dtype)
                    })
        
        display(pd.DataFrame(comparison))
        if not df_new.empty:
            print(f"\n--- SAMPLE DATA NEW (2 Baris) ---")
            display(df_new.head(2))
    else:
        print(f"⚠️ Tabel {new_t} kosong.")

📊 RINGKASAN MIGRASI (RECORDS COUNT)


,Tabel Lama,Tabel Baru,Old Recs,New Recs,Diff,Status
0,siswa,siswa,1469,1469,0,✅ OK
1,siswa_keluar,siswa_keluar,556,556,0,✅ OK
2,mitra,mitra,22,22,0,✅ OK
3,mitra_note,mitra_progres,296,296,0,✅ OK
4,mitra_users,kemitraan_verifikator,228,228,0,✅ OK
5,siswamitra,siswa_mitra,0,0,0,✅ OK
6,siswa_keluar_mitra,siswa_mitra_keluar,0,0,0,✅ OK



📢 TOTAL REKAPITULASI: 2571 (Old) ➔ 2571 (New)
✅ SEMUA DATA TERANGKUT

🔍 PERBANDINGAN TIPE DATA SIDE-BY-SIDE

=============== SISWA ➔ SISWA ===============


,Old Column,Old Type,➔,New Column,New Type
0,idsiswa,object,➔,id_siswa,object
1,tgl_daftar,object,➔,tanggal_registrasi,object
2,domisili,object,➔,domisili,object
3,nama_lengkap,object,➔,nama_lengkap,object
4,panggilan,object,➔,nama_panggilan,object
5,jkel,object,➔,jenis_kelamin,object
6,nama_sekolah,object,➔,asal_sekolah,object
7,level_sekolah,object,➔,tingkat_sekolah,object
8,nama_ortu,object,➔,nama_orang_tua,object
9,pekerjaan_ortu,object,➔,pekerjaan_orang_tua,object



--- SAMPLE DATA NEW (2 Baris) ---


...


=============== MITRA_USERS ➔ KEMITRAAN_VERIFIKATOR ===============


,Old Column,Old Type,➔,New Column,New Type
0,idmusers,object,➔,id_kemitraan,object
1,idmnote,object,➔,id_progres_mitra,object
2,idusers,object,➔,id_user,object



--- SAMPLE DATA NEW (2 Baris) ---


,id_kemitraan,id_progres_mitra,id_user
0,P00005,N00007,U00011
1,P00006,N00008,U00011



=============== SISWAMITRA ➔ SISWA_MITRA ===============
⚠️ Tabel siswa_mitra kosong.

=============== SISWA_KELUAR_MITRA ➔ SISWA_MITRA_KELUAR ===============
⚠️ Tabel siswa_mitra_keluar kosong.


## 4. Export ke Pickle

In [6]:
file_name = 'fase_4_hanif.pkl'

# Fix: Konversi tipe data StringDtype ke object agar kompatibel dengan Python 3.13 pickle
for table in transformed_dfs:
    df = transformed_dfs[table]
    if not df.empty:
        for col in df.columns:
            if str(df[col].dtype) in ['string', 'string[python]']:
                df[col] = df[col].astype(object)

pd.to_pickle(transformed_dfs, file_name)

total_records_new = sum(len(df) for df in transformed_dfs.values())
total_records_old = sum(len(records) for records in raw_data.values())

migration_result = {
    'fase': 'fase_4',
    'script': 'script_hanif',
    'fase_num': 4,
    'status': 'ready_for_insert',
    'old_records_total': total_records_old,
    'new_records_total': total_records_new,
    'diff': total_records_new - total_records_old,
    'pickle_file': file_name,
    'timestamp': datetime.now().isoformat()
}
print(json.dumps(migration_result, indent=2))

cursor_old.close()
cursor_new.close()
db_old.close()
db_new.close()

{
  "fase": "fase_4",
  "script": "script_hanif",
  "fase_num": 4,
  "status": "ready_for_insert",
  "old_records_total": 2571,
  "new_records_total": 2571,
  "diff": 0,
  "pickle_file": "fase_4_hanif.pkl",
  "timestamp": "2026-04-29T11:12:35.877642"
}
